# How few veto channels does the ULEG L1 need? Channel budgets for the trigger-pattern : veto-pattern map

The L1 of the ULEG trigger (thesis, section 4.2.2) vetoes an event when a PSD tile fires inside the region of interest (RoI) of its 3IR patterns. Implemented literally,
that is one veto line per 3IR pattern listening to its own set of tiles. The thesis already noted that this is not buildable as it stands and proposed to
*"predefine groups of PSD tiles and associate them with subsets of 3IR patterns"*, for instance one veto decision per detector face extended by up to two rows
of the neighbouring faces. This notebook makes that statement quantitative:

1. the notation, taken from the thesis where it exists, and the mathematics;
2. the data: the RoI map regenerated with HerdSoftware for every start plane, the trigger selection, the PSD rates and the measured backsplash;
3. 3D sketches of the RoIs and of the event RoI of Eq. (4.2);
4. the thesis' own strategies (full PSD, face + n rows) evaluated on Geant4 photons;
5. the L1 efficiency as a function of the number of channels, for optimised maps;
6. solutions constrained to the thesis' layout, and the program `tools/findChannelMap.py` that finds and lists a map for any budget.

Run time: about 30 minutes (the budget grid takes most of it); with `BICLUSTERING_USE_CACHE=1` the saved maps in `examples/results/` are reused.

In [ ]:
import os, sys, pickle, pathlib
sys.path.insert(0, "../analytic"); sys.path.insert(0, "../stochastic")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, ListedColormap
from matplotlib.patches import Patch

import patternDump as pd
import losslessUtils as lu
import mergeUtils as mu
import geometryUtils as gu
import plotUtils as pu
import rateUtils as ru
import photonEvents as pe

pu.setStyle()
blues = LinearSegmentedColormap.from_list("blues", pu.sequentialRamp)
resultsDirectory = pathlib.Path("results"); resultsDirectory.mkdir(exist_ok=True)
useCache = os.environ.get("BICLUSTERING_USE_CACHE") == "1"

def cached(name, compute):
    path = resultsDirectory / f"{name}.pkl"
    if useCache and path.exists():
        return pickle.loads(path.read_bytes())
    value = compute()
    path.write_bytes(pickle.dumps(value))
    return value

## 1. Notation and mathematics

The thesis notation is kept wherever it exists; the new symbols avoid the letters the thesis already uses for something else.

| Symbol | Meaning | Where it comes from |
|---|---|---|
| 3IR pattern $i$, $j$ | a three-in-a-row trigger pattern; $i$ in the even view, $j$ in the odd view of the same double layer | thesis, section 4.2.1 and Eq. (4.1) |
| $\{T\}_i$ | the RoI of pattern $i$: the PSD tiles partly or wholly inside its frustum | thesis, section 4.2.2 |
| $\{T\}_{\rm event}$ | $\big(\bigcup_i \{T\}^{\rm even}_i\big)\cap\big(\bigcup_j \{T\}^{\rm odd}_j\big)$ | thesis, Eq. (4.2) |
| $\varepsilon_{\rm L1}$ | L1 efficiency on the fiducial photons | thesis, Eq. (4.3), Table 4.2 |
| $M_{it}$ | $1$ if tile $t\in\{T\}_i$: the RoIs as a boolean matrix | new |
| $g(i)$, $N_G$ | the pattern group of $i$ (one veto decision per group); their number | new |
| $c(t)$, $N_C$ | the PSD channel of tile $t$ (an OR of tiles); their number | new |
| $S_{kl}$ | routing: channel $l$ vetoes group $k$ ($N_G\times N_C$, boolean) | new |
| $\{V\}_i$, $V_{it}=S_{g(i)\,c(t)}$ | the veto mask that a channel map gives pattern $i$ | new |
| $\xi$, $\kappa$ | mask excess and leak | new |
| $b_t$, $r_t$, $\tau$, $q_t$, $\pi_t$ | backsplash probability, activation rate, coincidence window, accidental and total veto probability of tile $t$ | new |

Budgets are $N_G$, $N_C$ rather than $K$, $L$ because $L$ would collide with the trigger levels L0-L2; "RoI" as in the thesis' list of acronyms; $\xi$, $\kappa$
because $\eta$, $\rho$, $\Delta$, $f$, $\lambda$ are taken.

**Channel map.** Patterns are merged into $N_G$ groups and tiles into $N_C$ channels (both partitions), and the routing $S$ says which channels veto which group;
pattern $i$ is then vetoed by $V_{it}=S_{g(i)c(t)}$ instead of $M_{it}$. Because $V_{it}$ depends on $t$ only through $c(t)$, an AND per channel realises the
tile-level intersection of Eq. (4.1) exactly, $\bigvee_t h_t V_{it}V_{jt}=\bigvee_l h_l S_{g(i)l}S_{g(j)l}$ with $h_l$ the OR of the channel's tiles, so the thesis'
factorisation (Eq. 4.2) carries over channel by channel.

**0% loss.** $V=M$ requires identical RoIs within a group and identical RoI memberships within a channel: $N_G\ge N_G^*$ = distinct rows of $M$, $N_C\ge N_C^*$ = distinct columns,
both reached together by grouping duplicates. With fan-out of a tile into several channels the minimum is the Boolean rank of $M$.

**Below the minimum.** Union routing ($S_{kl}=1$ if any tile of $l$ is in any RoI of $k$) keeps every RoI tile ($\kappa=0$) and adds tiles: the excess
$\xi=\sum (V-M)/\sum M$, or $\xi_{\rm ev}$ on the event RoIs. Every added tile costs photons in two ways: backsplash of the photon's own shower
($b_t$, strongly energy dependent) and accidental hits of the ~190 Hz per tile background within the coincidence window, $q_t = 1-e^{-r_t A\tau}$.
The optimiser minimises $\sum_{it}(V-M)_{it}\,C_{it}$ with $C_{it}=-\ln(1-\pi_t)$, $\pi_t = 1-(1-b_t)(1-q_t)$, the log-loss of independent tiles; the
**figure of merit is measured on events**: the fraction of L0-triggered fiducial Geant4 photons that pass L1 with the map, with the accidentals folded in.

## 2. The data

**RoIs.** The pattern $\to$ tile map is regenerated with HerdSoftware 0.3.2 (the version that still reads the compact-v2 datacard), the geometry card of the
AeffCalc proton study, and the RoI code of July 2024 that produced the thesis results; every start plane and every code (data/README.md, `data/regeneration/`).
The February 2023 dump (`data/patterns.txt`, start planes 8-9) came from earlier code and is kept for reference; the regenerated RoIs are tighter.

**Trigger selection.** The thesis' 3IR definition: a calorimeter cube in the RoI (CALO intersect), the third hit in the module adjacent to the first (the
zig-zags 546, 564 and the steep 544, 566 removed, as the analysis card's ignoreIDs), and no upward-pointing patterns (Table 4.1, "No Upwards"): upgoing tracks,
the particle moving towards the top, i.e. from below. Upward-pointing patterns are identified from the RoIs themselves: on the side faces the even planes
measure height, and the patterns whose net mat step is negative have RoIs low on the face that never reach TOP.

In [ ]:
allPatterns = pd.parsePatternDump()
legacy = pd.parsePatternDump(pd.legacyDumpPath)
print(f"{'selection':13} {'patterns':>8} {'upward':>7} {'N_G*':>6} {'N_C*':>6}")
for name, selection in pd.triggerSelections.items():
    chosen = pd.selectPatterns(allPatterns, **selection)
    sub = pd.buildIncidence(chosen).withoutEmptyColumns()
    exact = lu.findLosslessChannelMap(sub.matrix)
    print(f"{name:13} {len(chosen):8d} {sum(pd.pointsUpwards(p) for p in chosen):7d} {exact.routing.shape[0]:6d} {exact.routing.shape[1]:6d}")

side = [p for p in allPatterns if p.fitSide != "TOP" and p.firstPlane % 2 == 0]
print("\nside faces, even (height) planes: net mat step -> mean RoI row on the own face (0 = top edge, 8 = bottom), share of RoIs reaching TOP")
for step in (-2, -1, 0, 1, 2):
    group = [p for p in side if sum(p.matSteps) == step and any(f == p.fitSide for f, l, t in p.tiles)]
    rows = np.mean([np.mean([gu.gridIndex(f, t)[1] for f, l, t in p.tiles if f == p.fitSide]) for p in group])
    print(f"  {step:+d}: {rows:.2f}, {np.mean([any(f == 'TOP' for f, l, t in p.tiles) for p in group]):.0%}  ({len(group)} patterns, upward-pointing: {all(pd.pointsUpwards(p) for p in group)})")
print(f"\npatterns with an empty RoI: {[p.name for p in allPatterns if not p.tiles]} (all removed by the thesis selection)")

common = {p.name: p for p in legacy}
regenerated = {p.name: p for p in allPatterns}
shared = [n for n in common if n in regenerated]
print(f"2023 dump vs regenerated, {len(shared)} shared names: identical RoIs {sum(common[n].tiles == regenerated[n].tiles for n in shared)}, "
      f"mean RoI {np.mean([len(common[n].tiles) for n in shared]):.0f} -> {np.mean([len(regenerated[n].tiles) for n in shared]):.0f} tiles")

In [ ]:
patterns = pd.selectPatterns(allPatterns, **pd.triggerSelections["thesis"])
incidence = pd.buildIncidence(patterns)
M, tileKeys = incidence.matrix, incidence.tileKeys
pairBlocks = pd.evenOddBlocks(patterns)
fitFace = np.array([p.fitSide for p in patterns]); psdFace = np.array([key[0] for key in tileKeys])
faceIndex = np.array([pd.psdSides.index(f) for f in fitFace])
byName = {p.name: i for i, p in enumerate(patterns)}
print(f"thesis selection: {M.shape[0]} patterns x {M.shape[1]} tiles, {sum(len(e) * len(o) for e, o in pairBlocks)} even-odd pairs in {len(pairBlocks)} double layers")

**PSD rates.** The per-tile activation rates of thesis Fig. 4.9 (all species, compact-v2 PSD with 10 x 10 cm tiles), recovered from the published image by
`tools/extractPsdRates.py`, since the events are gone; their mean, spread and maximum agree with the summary computed from the events (AeffCalc `max_rates.txt`)
to 0.5%, 3% and 0.2%. **Backsplash.** Measured per tile on the Geant4 photons that survive locally (`stochastic/photonEvents.py`, `tools/measureBacksplash.py`): the
fraction of L0-triggered fiducial photons in which the tile fires, reweighted to an $E^{-2}$ spectrum.

In [ ]:
rates = dict(zip(tileKeys, ru.loadTileRates(tileKeys)))
backsplash = dict(zip(tileKeys, ru.loadBacksplash(tileKeys)))
figure, axes = plt.subplots(1, 4, figsize=(20, 5.4), constrained_layout=True)
rateNorm = Normalize(min(rates.values()), max(rates.values())); splashNorm = Normalize(0, max(backsplash.values()))
for layer in (0, 1):
    rateCollection = pu.plotUnfoldedPsd(axes[layer], rates, layer, rateNorm, blues, f"activation rate, layer {layer}")
    splashCollection = pu.plotUnfoldedPsd(axes[2 + layer], backsplash, layer, splashNorm, blues, f"backsplash per photon ($E^{{-2}}$), layer {layer}")
figure.colorbar(rateCollection, ax=axes[:2], shrink=0.8, label=r"$r_t$ [s$^{-1}$ cm$^{-2}$]")
figure.colorbar(splashCollection, ax=axes[2:], shrink=0.8, label=r"$b_t$")
plt.show()
r = ru.loadTileRates(tileKeys); b = ru.loadBacksplash(tileKeys)
for window in (100e-9, 1e-6):
    q = -np.expm1(-r * ru.tileArea * window)
    print(f"tau = {window * 1e9:4.0f} ns: accidental q_t mean {q.mean():.2e} per tile, against backsplash b_t mean {b.mean():.2e} (E^-2), "
          f"{ru.loadBacksplash(tileKeys, 'logUniform').mean():.2e} (log-uniform)")

## 3. RoIs and the event RoI in 3D

The PSD as the five faces of the cube, outer layer only, 10 cm tiles. The adjacency of the faces was inferred from the RoIs and agrees with the unfolding of
AeffCalc's psd_rates.py; real offsets, gaps and the stagger of the second layer are not drawn. Each figure shows the two opposite views.

In [ ]:
def drawPair(evenName, oddName):
    even, odd = M[byName[evenName]], M[byName[oddName]]
    sets = [(even & ~odd, pu.categoricalColours[0]), (odd & ~even, pu.categoricalColours[1]), (even & odd, pu.categoricalColours[6])]
    figure = plt.figure(figsize=(14, 5.2), constrained_layout=True)
    for n, azimuth in enumerate((-60, 120)):
        pu.plotPsdCube(figure.add_subplot(1, 2, n + 1, projection="3d"), pu.colourBySets(tileKeys, sets), azimuth=azimuth,
                       legend=[(sets[0][1], f"{evenName} (even view) only"), (sets[1][1], f"{oddName} (odd view) only"), (sets[2][1], "event RoI, Eq. (4.1)")] if n == 0 else None)
    figure.suptitle(f"RoIs of an even-odd pattern pair: {even.sum()} and {odd.sum()} tiles, event RoI {(even & odd).sum()} tiles", color=pu.inkPrimary)
    plt.show()

drawPair("TOP806555", "TOP906555")
drawPair("Xpos803555", "Xpos903555")
pairSizes = np.concatenate([(M[e].astype(float) @ M[o].T.astype(float)).ravel() for e, o in pairBlocks])
print(f"mean RoI of one pattern {M.sum(axis=1).mean():.0f} tiles; mean event RoI of a pair {pairSizes.mean():.0f} (min {pairSizes.min():.0f}, max {pairSizes.max():.0f})")

## 4. The thesis' strategies, evaluated on Geant4 photons

The full-PSD veto (one group, one channel), the face-segmented veto extended by $n$ rows of the adjacent faces (the thesis' hybrid; $n=2$ is its proposal), the
face-segmented veto with the union of the face's RoIs (its leak-free version), and the exact RoI veto. $\varepsilon_{\rm L1}$ is measured on the L0-triggered
fiducial photons, with accidental hits for a coincidence window $\tau$; the photons are weighted to $E^{-2}$ (with which the events give 0.90 for the full-PSD veto,
thesis Table 4.2: 0.89) or as generated, log-uniform in 10 MeV-100 GeV (as the energy-binned Fig. 4.5).

In [ ]:
photons = cached("photons", pe.loadPhotons)
triggered = pe.triggerPhotons(photons, patterns, tileKeys)
fiducial = [e for e in triggered if e["fiducial"]]
print(f"{len(photons)} photons with FIT hits, {len(triggered)} L0-triggered, {len(fiducial)} of them fiducial")
accidental = {window: ru.tileVetoProbability(tileKeys, window=window, backsplash=0.0) for window in (0.0, 100e-9, 1e-6)}

strategies = {"exact RoI": lu.findLosslessChannelMap(M),
              "full PSD veto": lu.ChannelMap(np.zeros(M.shape[0], int), np.zeros(M.shape[1], int), np.ones((1, 1), bool))}
for rows in range(4):
    strategies[f"face + {rows} rows"] = mu.channelMapFromMasks(M, faceIndex, np.array([gu.faceExtendedMask(tileKeys, f, rows) for f in pd.psdSides]))
strategies["face, union of RoIs"] = mu.channelMapFromMasks(M, faceIndex, np.array([M[faceIndex == k].any(axis=0) for k in range(5)]))

def summarise(channelMap, window=100e-9):
    mask = channelMap.vetoMask()
    event = mu.computePairMetrics(M, mask, pairBlocks)
    return dict(groups=channelMap.routing.shape[0], channels=channelMap.routing.shape[1], xiEvent=event["excess"], kappaEvent=event["leak"],
                kappa=mu.computeMaskCost(M, channelMap)[1] / M.sum(),
                epsE2=pe.l1Efficiency(triggered, mask, accidentalProbability=accidental[window], spectralIndex=2.0)[0],
                epsLog=pe.l1Efficiency(triggered, mask, accidentalProbability=accidental[window])[0],
                epsE2Long=pe.l1Efficiency(triggered, mask, accidentalProbability=accidental[1e-6], spectralIndex=2.0)[0],
                epsE2None=pe.l1Efficiency(triggered, mask, accidentalProbability=None, spectralIndex=2.0)[0])

strategyRows = {name: summarise(m) for name, m in strategies.items()}
print(f"\n{'strategy':20} {'N_G':>5} {'N_C':>5} {'xi_ev':>6} {'kappa':>7} {'kappa_ev':>9} | eps_L1 E^-2: {'no acc.':>8} {'100 ns':>7} {'1 us':>7} | log-uniform 100 ns")
for name, r in strategyRows.items():
    print(f"{name:20} {r['groups']:5d} {r['channels']:5d} {r['xiEvent']:6.2f} {r['kappa']:7.4f} {r['kappaEvent']:9.4f} |              {r['epsE2None']:8.3f} {r['epsE2']:7.3f} {r['epsE2Long']:7.3f} | {r['epsLog']:.3f}")

In [ ]:
bins = np.logspace(-2, 2, 9); centres = np.sqrt(bins[1:] * bins[:-1])
figure, drawingAxis = plt.subplots(figsize=(9, 5), constrained_layout=True)
for colour, name in zip(pu.categoricalColours, ("exact RoI", "face, union of RoIs", "face + 2 rows", "full PSD veto")):
    _, perBin, counts = pe.l1Efficiency(triggered, strategies[name].vetoMask(), bins, accidentalProbability=accidental[100e-9])
    error = np.sqrt(perBin * (1 - perBin) / np.maximum(counts, 1))
    drawingAxis.errorbar(centres, perBin, yerr=error, fmt="o-", color=colour, label=name, capsize=2)
drawingAxis.set(xscale="log", xlabel="photon energy [GeV]", ylabel=r"$\varepsilon_{\rm L1}$ (L0-triggered fiducial photons)", ylim=(0, 1.05),
                title=r"L1 efficiency against energy, accidentals in $\tau$ = 100 ns (compare thesis Fig. 4.5)")
drawingAxis.legend(loc="lower left")
plt.show()
print("fiducial photons per bin:", list(counts))

## 5. L1 efficiency against the number of channels

Optimised maps (greedy in two orders, the better of the two, then local refinement), cost from the measured backsplash and accidentals in 100 ns, on a grid of
budgets for all five FIT faces together; the thesis' strategies as points. Every optimised map uses union routing, so no RoI tile is ever dropped.

In [ ]:
vetoProbability = ru.tileVetoProbability(tileKeys, window=100e-9)
cost = ru.costMatrix(M.shape[0], vetoProbability)
channelBudgets = [1326, 512, 256, 128, 64, 32, 17, 8, 4, 2, 1]
groupBudgets = [1126, 256, 64, 16, 5, 1]

def optimiseGrid():
    scans = [mu.scanBudgets(M, channelBudgets, groupBudgets, cost=cost, order=order)[0] for order in ("channelsFirst", "groupsFirst")]
    maps = {}
    for budget in scans[0]:
        candidate = min((scan[budget] for scan in scans), key=lambda m: mu.computeMaskCost(M, m, cost)[0])
        if budget[0] <= 64 and budget[1] <= 256:
            candidate = mu.refineLocally(M, candidate, cost)
        maps[budget] = candidate
    return maps

optimised = cached("optimisedGrid", optimiseGrid)
optimisedRows = {budget: summarise(m) for budget, m in optimised.items()}
print("eps_L1 (E^-2, 100 ns) of the optimised maps (rows N_G, columns N_C):")
print("N_G \\ N_C " + "".join(f"{c:>7d}" for c in channelBudgets))
for g in groupBudgets:
    print(f"{g:>9d} " + "".join(f"{optimisedRows[(g, c)]['epsE2']:7.3f}" for c in channelBudgets))

In [ ]:
thesisPoints = {"full PSD veto": ("s", (7, 5)), "face + 2 rows": ("D", (7, 6)), "face, union of RoIs": ("o", (7, -14)), "exact RoI": ("*", (-10, 8))}
figure, axes = plt.subplots(1, 3, figsize=(20, 5.6), constrained_layout=True)
panels = ((axes[0], "epsE2", r"$\varepsilon_{\rm L1}$, $E^{-2}$ photons", "L1 efficiency, $E^{-2}$ spectrum, $\\tau$ = 100 ns"),
          (axes[1], "epsLog", r"$\varepsilon_{\rm L1}$, log-uniform photons", "L1 efficiency, as generated (10 MeV-100 GeV)"),
          (axes[2], "xiEvent", r"event-RoI excess $\xi_{\rm ev}$", "Extra tiles in the event veto regions"))
for drawingAxis, key, label, title in panels:
    for colour, g in zip(pu.ordinalRamp, groupBudgets):
        drawingAxis.plot(channelBudgets, [optimisedRows[(g, c)][key] for c in channelBudgets], "-", color=colour, label=f"$N_G$ = {g}" + (" ($N_G^*$)" if g == groupBudgets[0] else ""))
    for name, (marker, offset) in thesisPoints.items():
        r = strategyRows[name]
        drawingAxis.plot(r["channels"], r[key], marker, color=pu.inkPrimary, markersize=8, linestyle="none", markerfacecolor="none" if marker == "o" else pu.inkPrimary)
        drawingAxis.annotate(f"{name} ($N_G$={r['groups']})", (r["channels"], r[key]), xytext=offset, textcoords="offset points", fontsize=9,
                             color=pu.inkPrimary, ha="right" if offset[0] < 0 else "left")
    drawingAxis.set(xscale="log", xlabel="PSD channels $N_C$", ylabel=label, title=title)
axes[2].set(yscale="log")
axes[0].legend(title="pattern groups", loc="lower right")
plt.show()

In [ ]:
grid = np.array([[optimisedRows[(g, c)]["epsE2"] for c in channelBudgets] for g in groupBudgets])
figure, drawingAxis = plt.subplots(figsize=(11, 3.8), constrained_layout=True)
image = drawingAxis.imshow(grid, cmap=blues, aspect="auto")
for (row, column), value in np.ndenumerate(grid):
    drawingAxis.text(column, row, f"{value:.3f}", ha="center", va="center", fontsize=8, color="white" if value > grid.min() + 0.6 * (grid.max() - grid.min()) else pu.inkPrimary)
drawingAxis.set(xticks=range(len(channelBudgets)), xticklabels=channelBudgets, yticks=range(len(groupBudgets)), yticklabels=groupBudgets,
                xlabel="PSD channels $N_C$", ylabel="pattern groups $N_G$", title=r"$\varepsilon_{\rm L1}$ ($E^{-2}$ photons, $\tau$ = 100 ns) of the optimised maps")
drawingAxis.grid(False)
figure.colorbar(image, ax=drawingAxis, shrink=0.9)
plt.show()

## 6. Solutions constrained to the thesis' layout

Hardware segmented per face: groups may not mix patterns of different FIT faces, and possibly channels may not mix tiles of different PSD faces (one board per
face). `mu.findChannelMap` takes these as labels that no merge or move may cross, and tries a one-group-per-face seed besides the greedy orders.

In [ ]:
faceSeed = strategies["face, union of RoIs"]
def constrainedSolutions():
    solutions = {}
    for budget in ((5, 17), (5, 8)):
        for name, rowLabels, colLabels in (("groups per FIT face", fitFace, None), ("groups per FIT face, channels per PSD face", fitFace, psdFace)):
            best, candidates = mu.findChannelMap(M, *budget, cost=cost, rowLabels=rowLabels, colLabels=colLabels,
                                                 seeds=[("one group per FIT face", faceSeed)], orders=("groupsFirst",))
            solutions[(budget, name)] = (best, candidates)
    return solutions

constrained = cached("constrained", constrainedSolutions)
for (budget, name), (best, candidates) in constrained.items():
    r = summarise(best)
    print(f"N_G={budget[0]}, N_C={budget[1]}, {name}: found ({r['groups']}, {r['channels']}), eps_L1 E^-2 {r['epsE2']:.3f}, log-uniform {r['epsLog']:.3f}, "
          f"xi_ev {r['xiEvent']:.2f}; candidates " + ", ".join(f"{n} {c / (cost * M).sum():.2f}" for n, c in candidates))

In [ ]:
chosen = constrained[((5, 8), "groups per FIT face, channels per PSD face")][0]
groupFace = [fitFace[np.flatnonzero(chosen.rowGroup == k)[0]] for k in range(chosen.routing.shape[0])]
channelOf = {key: chosen.colGroup[column] for column, key in enumerate(tileKeys)}
figure, axes = plt.subplots(1, 2, figsize=(13, 6.2), constrained_layout=True)
for layer in (0, 1):
    pu.plotUnfoldedPsd(axes[layer], channelOf, layer, Normalize(-0.5, 7.5), ListedColormap(pu.categoricalColours), f"PSD layer {layer} (TOP in the centre)")
figure.legend(handles=[Patch(facecolor=pu.categoricalColours[l], label=f"C{l} -> " + " ".join(groupFace[k] for k in np.flatnonzero(chosen.routing[:, l])))
                       for l in range(chosen.routing.shape[1])], loc="outside right center", title="channel -> patterns of the FIT faces it vetoes")
figure.suptitle(f"Constrained solution, N_G = 5 (one per FIT face), N_C = {chosen.routing.shape[1]} (each within one PSD face)", color=pu.inkPrimary)
plt.show()
listing = mu.describeChannelMap(chosen, patterns, tileKeys).splitlines()
print("\n".join(line if len(line) < 160 else line[:157] + "..." for line in listing if line.startswith(("C", "G")) and ("tiles:" in line or "<-" in line)))

## 7. The generic program

`tools/findChannelMap.py` runs the same search for any budget, trigger selection, coincidence window and constraint set, and lists the combination (patterns per
group, tiles per channel, routing) as text and JSON; `--events` adds the L1 efficiency measured on the Geant4 photons:

```
python tools/findChannelMap.py --groups 16 --channels 64 --events                          # optimise, all FIT faces
python tools/findChannelMap.py --groups 5 --channels 17 --groups-per-fit-face \
                               --channels-per-psd-face --window 1000 --json map.json       # per-face layout, 1 us window
python tools/findChannelMap.py --strategy face-rows --rows 2 --events                      # evaluate the thesis' hybrid
```

## 8. Caveats

- The photon sample is small: 340 L0-triggered fiducial photons, from the upper hemisphere only. Efficiencies are good to a few percent, the per-tile
  backsplash map is smoothed, and side-face photons are under-represented.
- The coincidence window is a placeholder; the accidentals scale linearly with it. Background leak is measured only geometrically ($\kappa$): there are no
  charged-particle events left to measure it with.
- The calorimeter requirement is applied per pattern, not per pair, and L2 (the calorimeter threshold) is not applied.
- The regenerated RoIs come from HerdSoftware 0.3.2 and the 2024 RoI code; the 2023 dump differs from them, and the configuration that made it could not
  be reproduced.
- The search finds good local optima, not certified ones.